# Эксперименты ЛИМС на A100
Результаты запуска lims_gpu_20260914_v1. Train: 2023–2024; validation: 2025; test: 2026. Выбор модели только по validation. Горизонт относительно неизвестного по смыслу timestamp ЛИМС. См. AGENT_TRAINING_HANDOFF.md. Запустите все ячейки для таблиц и графиков; обучение запускается отдельно скриптом train_lims_a100.py.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display
HERE = Path.cwd().resolve()
EDA = HERE if HERE.name == 'eda' else HERE / 'eda'
RUN = EDA / 'experiments' / 'lims_gpu_20260914_v1'
metrics = pd.read_csv(RUN / 'metrics.csv')
predictions = pd.read_csv(RUN / 'predictions.csv', parse_dates=['timestamp'])
display(pd.read_csv(RUN / 'target_inventory.csv'))
display(metrics[(metrics['split']=='test') & (metrics['selected'] | metrics['model'].isin(['train_median','last_lab_delay24h']))])

,target,horizon,train,validation,test
0,Mg.Sulfur,0,792,416,248
1,Mg.Sulfur,6,792,416,248
2,D15,0,691,181,139
3,D15,6,691,181,139
4,CetaneNumber,0,25,10,6
5,CetaneNumber,6,25,10,6
6,FlashPoint,0,884,401,228
7,FlashPoint,6,884,401,228


,target,horizon,model,split,selected,n,mae,median_ae,rmse
3,Mg.Sulfur,0,catboost_d4_MAE,test,True,248,1.323287,0.897659,1.997736
9,Mg.Sulfur,0,train_median,test,False,248,1.671371,1.100000,2.491720
11,Mg.Sulfur,0,last_lab_delay24h,test,False,248,1.968145,1.200000,2.971878
17,Mg.Sulfur,6,catboost_d6_RMSE,test,True,248,2.679444,1.115659,8.390792
21,Mg.Sulfur,6,train_median,test,False,248,1.671371,1.100000,2.491720
23,Mg.Sulfur,6,last_lab_delay24h,test,False,248,2.007258,1.200000,2.965324
29,D15,0,catboost_d6_RMSE,test,True,139,2.629324,1.799075,3.762026
33,D15,0,train_median,test,False,139,2.355396,1.299988,3.838262
35,D15,0,last_lab_delay24h,test,False,139,1.694243,0.799988,2.926424
41,D15,6,catboost_d6_RMSE,test,True,139,2.698702,1.942276,3.862535


In [2]:
shown = metrics[(metrics['split']=='test') & (metrics['selected'] | metrics['model'].isin(['train_median','last_lab_delay24h']))].copy()
fig = px.bar(shown, x='model', y='mae', color='model', facet_row='target', facet_col='horizon', title='MAE на 2026 годе: модель и baseline', height=1100)
fig.update_yaxes(matches=None)
fig.show()

In [3]:
for (target, horizon), part in predictions[predictions['split']=='test'].groupby(['target','horizon']):
    fig = px.line(part, x='timestamp', y=['actual','prediction'], title=f'{target}, горизонт {horizon} ч: ЛИМС и прогноз')
    fig.show()

In [4]:
test = predictions[predictions['split']=='test'].copy()
test['absolute_error'] = (test.actual-test.prediction).abs()
test['month'] = test.timestamp.dt.to_period('M').astype(str)
monthly = test.groupby(['target','horizon','month'],as_index=False).absolute_error.mean()
px.line(monthly,x='month',y='absolute_error',color='horizon',facet_row='target',height=900,title='Дрейф ошибок по месяцам').update_yaxes(matches=None).show()

## Интерпретация
Сравнивайте с обоими baseline. Низкая MAE не доказывает обнаружение редких превышений, причинный эффект управления или пригодность для пуска. Плотность W70/F30 АВТ не является D15 готового продукта. Численные ограничения, цены, задержки лаборатории и параметры сценариев должны иметь пометку происхождения.